# Categorie Getgems dei numeri (+888)

Questo notebook assegna a ogni numero **tutte** le categorie che Getgems gli dà (34 categorie, tutte basate su regole delle cifre),
le verifica con i controlli fatti sul sito e salva il dataset con le nuove colonne.

* **input**: `dataset_clean/fragment_numbers_sample_500_clean.csv`, il dataset pulito prodotto da `pulizia_history.ipynb`;
* **elenco delle categorie con gli ID di Getgems**: `categories/getgems_categories.csv`;
* **verifiche fatte sul sito**: `categories/verifica_numeri/`;
* **output**: `dataset_clean_categories/fragment_numbers_sample_500_clean_categories.csv`.

Le categorie degli username le calcola `unisci_categorie.py`.

## 0. Parametri

In [1]:
import json, re, calendar
from pathlib import Path
import pandas as pd

BASE = Path('/Users/cesareanzellotti/Documents/CNA')
INPUT = BASE / 'dataset_clean' / 'fragment_numbers_sample_500_clean.csv'
CATEGORIES = BASE / 'categories' / 'getgems_categories.csv'
VERIFY_DIR = BASE / 'categories' / 'verifica_numeri'
OUTPUT = BASE / 'dataset_clean_categories' / 'fragment_numbers_sample_500_clean_categories.csv'

pd.set_option('display.max_colwidth', 70)
pd.set_option('display.width', 200)

## 1. Le regole

Getgems assegna ai numeri 34 categorie, tutte basate su regole delle cifre. Le regole qui sotto sono state ricavate dagli esempi
delle pagine delle categorie e poi verificate sul sito (sezione 3). `s` sono le cifre dopo il prefisso `888`: 8 nei numeri
normali, 4 nei numeri corti `+888 8 xxx`.

* **Ripetizioni**: almeno N cifre uguali di fila. Le categorie sono cumulative: `0000` sta in 2, 3 e 4 Digits.
* **Schemi** (ABAB, AABB, ABCABC…): in qualsiasi punto, con lettere = cifre diverse. In AxAxA la `x` è una cifra qualsiasi.
* **Club** (69, 666, 777, 888, 520, 168, 1314, 123, 007) e **Pi Club** (`31415`): `s` contiene la sequenza.
* **Palindromi**: valgono solo per i numeri a 8 cifre. "Beginning 888" e "With 888 On Each Side" = tutto il numero, 888 compreso, è palindromo.
* **Date Club**: il primo o il secondo blocco da 4 cifre è una data valida MMGG. GGMM non conta, e nemmeno una data a cavallo dei blocchi.
* **Years 1900 2015**: le ultime 4 cifre sono un anno tra 1900 e 2015.
* **Numbers Length 7**: numero corto **che non ha nessun'altra categoria**. Per esempio `+888 8 177` sta solo in Repeating 2 Digits.

In [2]:
def _mmdd(b):
    # True se 4 cifre formano una data valida MMGG (29 febbraio compreso)
    m, d = int(b[:2]), int(b[2:])
    return 1 <= m <= 12 and 1 <= d <= calendar.monthrange(2024, m)[1]

def _pal(x):
    return x == x[::-1]

# s = cifre dopo il prefisso 888, full = tutte le cifre del numero
NUMBER_RULES = {
    'Numbers Repeating 2 Digits': lambda s, full: re.search(r'(\d)\1', s),
    'Numbers Repeating 3 Digits': lambda s, full: re.search(r'(\d)\1{2}', s),
    'Numbers Repeating 4 Digits': lambda s, full: re.search(r'(\d)\1{3}', s),
    'Numbers Repeating 5 Digits': lambda s, full: re.search(r'(\d)\1{4}', s),
    'Numbers Repeating 6 Digits': lambda s, full: re.search(r'(\d)\1{5}', s),
    'Numbers Repeating 7 Digits': lambda s, full: re.search(r'(\d)\1{6}', s),
    'Numbers Repeating 8 Digits': lambda s, full: re.search(r'(\d)\1{7}', s),
    'Numbers Repeating ABAB':     lambda s, full: re.search(r'(\d)(?!\1)(\d)\1\2', s),
    'Numbers Repeating AABB':     lambda s, full: re.search(r'(\d)\1(?!\1)(\d)\2', s),
    'Numbers Repeating ABABAB':   lambda s, full: re.search(r'(\d)(?!\1)(\d)\1\2\1\2', s),
    'Numbers Repeating ABCABC':   lambda s, full: re.search(r'(\d)(?!\1)(\d)(?!\1|\2)(\d)\1\2\3', s),
    'Numbers Repeating AABBCC':   lambda s, full: re.search(r'(\d)\1(?!\1)(\d)\2(?!\1|\2)(\d)\3', s),
    'Numbers Repeating ABCDABCD': lambda s, full: re.search(r'(\d)(?!\1)(\d)(?!\1|\2)(\d)(?!\1|\2|\3)(\d)\1\2\3\4', s),
    'Numbers Repeating AxAxA':    lambda s, full: re.search(r'(\d)\d\1\d\1', s),
    'Numbers Repeating AxAxAxA':  lambda s, full: re.search(r'(\d)\d\1\d\1\d\1', s),
    '69 Club':   lambda s, full: '69' in s,
    '666 Club':  lambda s, full: '666' in s,
    '777 Club':  lambda s, full: '777' in s,
    '888 Club':  lambda s, full: '888' in s,
    '520 Club':  lambda s, full: '520' in s,
    '168 Club':  lambda s, full: '168' in s,
    '1314 Club': lambda s, full: '1314' in s,
    '123 Club':  lambda s, full: '123' in s,
    '007 Club':  lambda s, full: '007' in s,
    'Pi Club':   lambda s, full: '31415' in s,
    # i palindromi valgono solo per i numeri normali (8 cifre dopo 888)
    'Palindromic Numbers':                       lambda s, full: len(s) == 8 and _pal(s),
    'Palindromic Numbers Without First':         lambda s, full: len(s) == 8 and _pal(s[1:]),
    'Palindromic Numbers Without Last':          lambda s, full: len(s) == 8 and _pal(s[:-1]),
    'Palindromic Numbers Beginning 888':         lambda s, full: len(s) == 8 and _pal(full),
    'Palindromic Numbers With 888 On Each Side': lambda s, full: len(s) == 8 and _pal(full),
    'Numbers With 888 On Each Side': lambda s, full: s.endswith('888'),
    'Date Club':        lambda s, full: len(s) == 8 and (_mmdd(s[:4]) or _mmdd(s[4:])),
    'Years 1900 2015':  lambda s, full: len(s) == 8 and 1900 <= int(s[-4:]) <= 2015,
}
NUMBER_CATEGORIES = list(NUMBER_RULES) + ['Numbers Length 7']

def number_categories(name):
    full = re.sub(r'\D', '', name)
    s = full[3:]
    cats = [cat for cat, rule in NUMBER_RULES.items() if rule(s, full)]
    # "Numbers Length 7" solo per i numeri corti che non hanno nessun'altra categoria
    if len(full) == 7 and not cats:
        cats = ['Numbers Length 7']
    return cats

# qualche esempio
for n in ['+888 0777 7777', '+888 0814 3929', '+888 8 177', '+888 8 680', '+888 0768 4929']:
    print(f'{n:16s}', number_categories(n))

+888 0777 7777   ['Numbers Repeating 2 Digits', 'Numbers Repeating 3 Digits', 'Numbers Repeating 4 Digits', 'Numbers Repeating 5 Digits', 'Numbers Repeating 6 Digits', 'Numbers Repeating 7 Digits', 'Numbers Repeating AxAxA', 'Numbers Repeating AxAxAxA', '777 Club', 'Palindromic Numbers Without First']
+888 0814 3929   ['Date Club']
+888 8 177       ['Numbers Repeating 2 Digits']
+888 8 680       ['Numbers Length 7']
+888 0768 4929   []


## 2. Assegnazione delle categorie a ogni numero

In [3]:
# ID delle categorie sul sito di Getgems (es. Pi Club -> 160), presi da getgems_categories.csv
cat_ids = pd.read_csv(CATEGORIES).query("collezione == 'numbers'")
CATEGORY_ID = dict(zip(cat_ids.categoria, cat_ids.category_id.astype(int).tolist()))
assert set(NUMBER_CATEGORIES) == set(CATEGORY_ID), 'le categorie delle regole e quelle del CSV non coincidono'

numbers_cat = pd.read_csv(INPUT, index_col=0)
numbers_cat['categorie_getgems'] = numbers_cat['name'].map(number_categories)
numbers_cat['id_categorie_getgems'] = numbers_cat['categorie_getgems'].apply(lambda l: [CATEGORY_ID[c] for c in l])
numbers_cat['n_categorie'] = numbers_cat['categorie_getgems'].str.len()

print('numeri con almeno una categoria:', (numbers_cat.n_categorie > 0).sum(), 'su', len(numbers_cat))
print('quante categorie per numero:', numbers_cat.n_categorie.value_counts().sort_index().to_dict())
conteggi = pd.DataFrame({
    'id_getgems': pd.Series(CATEGORY_ID),
    'numeri': pd.Series([c for l in numbers_cat.categorie_getgems for c in l]).value_counts(),
}).reindex(NUMBER_CATEGORIES).fillna(0).astype(int)
conteggi

numeri con almeno una categoria: 2193 su 5000
quante categorie per numero: {0: 2807, 1: 1830, 2: 284, 3: 31, 4: 20, 5: 12, 6: 6, 7: 8, 9: 1, 10: 1}


,id_getgems,numeri
Numbers Repeating 2 Digits,200,132
Numbers Repeating 3 Digits,201,40
Numbers Repeating 4 Digits,140,24
Numbers Repeating 5 Digits,141,12
Numbers Repeating 6 Digits,142,6
Numbers Repeating 7 Digits,143,1
Numbers Repeating 8 Digits,144,0
Numbers Repeating ABAB,203,117
Numbers Repeating AABB,202,25
Numbers Repeating ABABAB,148,5


## 3. Verifica sul sito di Getgems

Le verifiche le ho fatte dal browser, guardando le pagine del sito come un normale utente. I risultati sono in due file:
* `verifica_pagine_nft_numeri.csv`: per 67 numeri, l'elenco **completo** delle categorie mostrato sulla pagina dell'NFT
  (50 estratti a caso dal campione, 25 con almeno una categoria e 25 senza, più i 16 numeri corti e `+888 0777 7777`);
* `verifica_categorie_numeri.csv`: una riga per ogni coppia (numero, categoria) controllata. Le fonti sono tre:
  1. **pagina NFT**: le categorie dei 67 numeri qui sopra;
  2. **ricerca nella categoria**: 98 numeri cercati nella pagina di una categoria, compresi alcuni "negativi difficili"
     (es. un numero che finisce con `88` ma non con `888`, l'anno 2041, un `649` per il 69 Club);
  3. **esempi delle pagine delle categorie**: i primi ~15 numeri di ogni categoria (509 in tutto) devono essere riconosciuti dalle regex.

La verifica ha già corretto due regole: i palindromi non valgono per i numeri corti, e "Numbers Length 7" è una categoria di riserva.

In [4]:
# 1) Pagine NFT: l'insieme di categorie previsto deve essere identico a quello mostrato da Getgems
pagine = pd.read_csv(VERIFY_DIR / 'verifica_pagine_nft_numeri.csv')
pagine['getgems'] = pagine.categorie_getgems.apply(json.loads)
pagine['regex'] = pagine.numero.map(number_categories)
pagine['identico'] = [set(a) == set(b) for a, b in zip(pagine.regex, pagine.getgems)]
print(f"pagine NFT con insieme di categorie identico: {pagine.identico.sum()} / {len(pagine)}",
      f"(di cui {(pagine.getgems.str.len() == 0).sum()} numeri senza nessuna categoria)")

# 2) Tutti i controlli (numero, categoria): pagine NFT + ricerche nelle categorie + esempi delle pagine delle categorie
ver = pd.read_csv(VERIFY_DIR / 'verifica_categorie_numeri.csv')
# la previsione si ricalcola con le regole attuali: se una regola cambia, la verifica si aggiorna da sola
ver['regex'] = [c in number_categories(n) for n, c in zip(ver.numero, ver.categoria)]
ver['esito'] = [('TP' if g else 'FP') if r else ('FN' if g else 'TN') for r, g in zip(ver.regex, ver.getgems)]
tp, fp, fn = (ver.esito == 'TP').sum(), (ver.esito == 'FP').sum(), (ver.esito == 'FN').sum()
print('controlli (numero, categoria):', len(ver), '|', ver.esito.value_counts().to_dict())
print(f'precision {tp / (tp + fp):.3f} | recall {tp / (tp + fn):.3f}')
print('categorie con almeno un controllo:', ver.categoria.nunique(), 'su', len(NUMBER_CATEGORIES))
pd.crosstab(ver.fonte, ver.esito)

pagine NFT con insieme di categorie identico: 67 / 67 (di cui 25 numeri senza nessuna categoria)
controlli (numero, categoria): 639 | {'TP': 629, 'TN': 10}
precision 1.000 | recall 1.000
categorie con almeno un controllo: 34 su 34


esito,TN,TP
fonte,,
esempio pagina categoria,0,500
pagina NFT,0,57
ricerca nella categoria,10,72


## 4. Salvataggio
`dataset_clean_categories/fragment_numbers_sample_500_clean_categories.csv` = il dataset pulito dei numeri più tre colonne:
* `categorie_getgems`: elenco JSON di **tutte** le categorie del numero, es. `["69 Club", "Date Club"]`, oppure `[]`;
* `id_categorie_getgems`: gli ID delle stesse categorie sul sito di Getgems, nello stesso ordine, es. `[167, 170]`.
  Il link alla categoria è `https://getgems.io/collection/<collezione>?categoryId=<id>`;
* `n_categorie`: quante sono.

In [5]:
OUTPUT.parent.mkdir(exist_ok=True)
cols = ['address', 'name', 'owner_address', 'categorie_getgems', 'id_categorie_getgems', 'n_categorie', 'history']
(numbers_cat.assign(categorie_getgems=numbers_cat.categorie_getgems.apply(json.dumps),
                    id_categorie_getgems=numbers_cat.id_categorie_getgems.apply(json.dumps))
            [cols].to_csv(OUTPUT))
print('salvato', OUTPUT)

# Rilettura di controllo
chk = pd.read_csv(OUTPUT, index_col=0)
print(chk.shape, list(chk.columns))
chk[chk.n_categorie > 1][['name', 'categorie_getgems', 'id_categorie_getgems', 'n_categorie']].head()

salvato /Users/cesareanzellotti/Documents/CNA/dataset_clean_categories/fragment_numbers_sample_500_clean_categories.csv
(5000, 7) ['address', 'name', 'owner_address', 'categorie_getgems', 'id_categorie_getgems', 'n_categorie', 'history']


,name,categorie_getgems,id_categorie_getgems,n_categorie
10,+888 0324 1931,"[""Date Club"", ""Years 1900 2015""]","[170, 169]",2
23,+888 0328 1935,"[""Date Club"", ""Years 1900 2015""]","[170, 169]",2
43,+888 0130 7692,"[""69 Club"", ""Date Club""]","[167, 170]",2
54,+888 0425 1937,"[""Date Club"", ""Years 1900 2015""]","[170, 169]",2
70,+888 0213 7569,"[""69 Club"", ""Date Club""]","[167, 170]",2
